In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:

import dask_cudf
df_parquet = dask_cudf.read_parquet("/content/drive/MyDrive/Full_Taxi_data/merged_taxi_data_v3.parquet")
len(df_parquet)

79465312

In [3]:
%load_ext cudf.pandas
import pandas as pd

In [4]:
columns_to_keep = [
    'tpep_pickup_datetime',
    'PULocationID',
    'temp_c',
    'rain_mm',
    'is_rain',
    'weather_code',
    'is_holiday'
]

In [5]:
df_parquet['pickup_count'] = 1

In [8]:
df_parquet.head(5)

,VendorID,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,...,rain_mm,is_rain,weather_code,trip_duration,pickup_hour,day_of_week,is_weekend,is_holiday,pickup_count,time_bucket_6H
0,2,2024-01-01 01:17:43,1,1.72,1,186,79,2,17.700001,1.0,...,0.0,0.0,3,19.800000,0,0,0,1,1,2024-01-01
1,1,2024-01-01 00:09:36,1,1.80,1,140,236,1,10.000000,3.5,...,0.0,0.0,3,6.600000,0,0,0,1,1,2024-01-01
2,1,2024-01-01 00:35:01,1,4.70,1,236,79,1,23.299999,3.5,...,0.0,0.0,3,17.916667,0,0,0,1,1,2024-01-01
3,1,2024-01-01 00:44:56,1,1.40,1,79,211,1,10.000000,3.5,...,0.0,0.0,3,8.300000,0,0,0,1,1,2024-01-01
4,1,2024-01-01 00:52:57,1,0.80,1,211,148,1,7.900000,3.5,...,0.0,0.0,3,6.100000,0,0,0,1,1,2024-01-01


In [ ]:
df_parquet['tpep_pickup_datetime'] = df_parquet['tpep_pickup_datetime'].astype('datetime64[ns]')

SIX_HOURS_IN_NANOS = 21600000000000

df_parquet['time_bucket_6H'] = (
    (df_parquet['tpep_pickup_datetime'].astype('int64') // SIX_HOURS_IN_NANOS) * SIX_HOURS_IN_NANOS
).astype('datetime64[ns]')

df_parquet = df_parquet.drop(columns=['tpep_pickup_datetime'])

In [ ]:

print("Preparing collection rules...")
agg_dict = {
    'pickup_count': 'sum',     
    'temp_c': 'mean',          
    'rain_mm': 'max',           
    'is_rain': 'max',         
    'weather_code': 'first',   
    'is_holiday': 'max'         
}

جاري تجهيز قواعد التجميع...


In [10]:
df_grouped = df_parquet.groupby(['PULocationID', 'time_bucket_6H']).agg(agg_dict).reset_index()

In [11]:
df_agg = df_grouped.compute()

In [12]:
df_agg = df_agg.sort_values(['PULocationID', 'time_bucket_6H']).reset_index(drop=True)

In [13]:
print("Compilation completed successfully! The new number of rows is:", len(df_agg))
print(df_agg.head())

تم التجميع بنجاح! عدد الصفوف الجديد هو: 561083
   PULocationID      time_bucket_6H  pickup_count  temp_c  rain_mm  is_rain  \
0             1 2024-01-01 12:00:00             2    6.30      0.0      0.0   
1             1 2024-01-02 18:00:00             2    0.80      0.0      0.0   
2             1 2024-01-03 12:00:00             2    5.95      0.0      0.0   
3             1 2024-01-04 12:00:00             1    6.30      0.0      0.0   
4             1 2024-01-05 12:00:00             1    0.10      0.0      0.0   

   weather_code  is_holiday  
0             3           1  
1             0           0  
2             1           0  
3             2           0  
4             0           0  


In [ ]:
total_trips = df_agg['pickup_count'].sum()
print(f"Total number of trips after aggregation: {total_trips:,.0f} trips")

print("\n--- Statistics of the number of trips (for each region in 6 hours) ---")
print(df_agg['pickup_count'].describe())

print("\n--- The 5 busiest areas and periods (naturally large numbers) ---")
busy_zones = df_agg.sort_values('pickup_count', ascending=False).head(5)
print(busy_zones[['PULocationID', 'time_bucket_6H', 'pickup_count']])

إجمالي عدد الرحلات الكلي بعد التجميع: 79,465,312 رحلة

--- إحصائيات عدد الرحلات (لكل منطقة في 6 ساعات) ---
count    561083.000000
mean        141.628444
std         344.068377
min           1.000000
25%           3.000000
50%           9.000000
75%          62.000000
max        3742.000000
Name: pickup_count, dtype: float64

--- أكثر 5 مناطق وفترات ازدحاماً (أرقام كبيرة طبيعية) ---
        PULocationID      time_bucket_6H  pickup_count
497716           237 2025-05-08 12:00:00          3742
334232           161 2025-11-19 18:00:00          3679
333032           161 2025-01-23 18:00:00          3671
333340           161 2025-04-10 18:00:00          3606
333308           161 2025-04-02 18:00:00          3602


In [ ]:
import numpy as np

df_agg_pd = df_agg.to_pandas() if hasattr(df_agg, 'to_pandas') else df_agg

print("1. Creating a Continuous Time Grid (Zero Padding)...")
# Get the absolute minimum and maximum time in the dataset
min_time = df_agg_pd['time_bucket_6H'].min()
max_time = df_agg_pd['time_bucket_6H'].max()

# Create an array of ALL possible 6-hour buckets between min and max time
all_times = pd.date_range(start=min_time, end=max_time, freq='6h')

# Get all unique Zone IDs
all_zones = df_agg_pd['PULocationID'].unique()

# Create a complete combination (MultiIndex) of every Zone and every Time Bucket
full_index = pd.MultiIndex.from_product(
    [all_zones, all_times],
    names=['PULocationID', 'time_bucket_6H']
)
df_grid = pd.DataFrame(index=full_index).reset_index()

print("2. Merging real data with the Time Grid...")
# Left join the grid with our actual aggregated data
df_padded = pd.merge(df_grid, df_agg_pd, on=['PULocationID', 'time_bucket_6H'], how='left')

print("3. Handling Missing Values (Filling Gaps)...")
# If a row was missing in real data, it means 0 pickups happened
df_padded['pickup_count'] = df_padded['pickup_count'].fillna(0)

# For weather and holidays, we fill missing gaps with the previous available record (Forward Fill)
weather_cols = ['temp_c', 'rain_mm', 'is_rain', 'weather_code', 'is_holiday']
df_padded[weather_cols] = df_padded.groupby('PULocationID')[weather_cols].ffill()
# Fallback Backward Fill (just in case the very first period was missing)
df_padded[weather_cols] = df_padded.groupby('PULocationID')[weather_cols].bfill()

print("4. Extracting Time Features...")
# Time features MUST be extracted after padding to ensure they are correct for the new rows
df_padded['pickup_hour'] = df_padded['time_bucket_6H'].dt.hour
df_padded['day_of_week'] = df_padded['time_bucket_6H'].dt.dayofweek
df_padded['is_weekend'] = (df_padded['day_of_week'] >= 5).astype('int8')

print("5. Engineering Lag and Rolling Features (Now 100% Accurate!)...")
# Lag features will now strictly look at the actual previous 6 hours, even if it was 0
df_padded['lag_1_6h'] = df_padded.groupby('PULocationID')['pickup_count'].shift(1)
df_padded['lag_2_6h'] = df_padded.groupby('PULocationID')['pickup_count'].shift(2)
df_padded['lag_4_6h'] = df_padded.groupby('PULocationID')['pickup_count'].shift(4)

df_padded['rolling_mean_24h'] = df_padded.groupby('PULocationID')['pickup_count'].transform(
    lambda x: x.rolling(window=4, min_periods=1).mean()
)

print("6. Creating Target Variable...")
# Target is the next strictly continuous 6-hour window
df_padded['target_next_6h'] = df_padded.groupby('PULocationID')['pickup_count'].shift(-1)

print("7. Cleaning up NaN values...")
# Drop boundaries (first few rows and last row per zone due to shift)
df_final = df_padded.dropna().reset_index(drop=True)

print("--------------------------------------------------")
print("Data Padding and Feature Engineering Completed Successfully!")
print(f"Final dataset shape: {df_final.shape}")

1. Creating a Continuous Time Grid (Zero Padding)...
2. Merging real data with the Time Grid...
3. Handling Missing Values (Filling Gaps)...
4. Extracting Time Features...
5. Engineering Lag and Rolling Features (Now 100% Accurate!)...
6. Creating Target Variable...
7. Cleaning up NaN values...
--------------------------------------------------
Data Padding and Feature Engineering Completed Successfully!
Final dataset shape: (735085, 16)


In [17]:
# Validation for Zone 1
zone_to_check = 1

check_df = df_final[df_final['PULocationID'] == zone_to_check].head(10)

cols_to_view = [
    'time_bucket_6H',
    'pickup_count',
    'lag_1_6h',
    'target_next_6h'
]

print(f"--- Data Validation for Zone {zone_to_check} ---")
print(check_df[cols_to_view])

--- Data Validation for Zone 1 ---
       time_bucket_6H  pickup_count  lag_1_6h  target_next_6h
0 2024-01-02 00:00:00           0.0       0.0             0.0
1 2024-01-02 06:00:00           0.0       0.0             0.0
2 2024-01-02 12:00:00           0.0       0.0             2.0
3 2024-01-02 18:00:00           2.0       0.0             0.0
4 2024-01-03 00:00:00           0.0       2.0             0.0
5 2024-01-03 06:00:00           0.0       0.0             2.0
6 2024-01-03 12:00:00           2.0       0.0             0.0
7 2024-01-03 18:00:00           0.0       2.0             0.0
8 2024-01-04 00:00:00           0.0       0.0             0.0
9 2024-01-04 06:00:00           0.0       0.0             1.0


In [ ]:


expected_target = df_final.groupby('PULocationID')['pickup_count'].shift(-1)

match_check = (df_final['target_next_6h'] == expected_target)

valid_rows = expected_target.notna()
accuracy = match_check[valid_rows].mean() * 100

if accuracy == 100.0:
    print("✅ Excellent! All rows checked. Target column (target_next_6h) is 100% identical to the future.")
else:
    print(f"❌ There is an error! The match percentage is only {accuracy}%.")


print("\n==================================================")
print("2. General Data Health")
print("==================================================")

#1. Checking for Missing Values
missing_values = df_final.isnull().sum()
if missing_values.sum() == 0:
    print("✅ There are no missing values ​​(NaNs) in the data.")
else:
    print("⚠️ Warning: Missing values:")
    print(missing_values[missing_values > 0])

#2. Examining Infinite Values
# Some mathematical operations may produce (inf), you must ensure that it does not exist
numeric_cols = df_final.select_dtypes(include=[np.number]).columns
inf_check = np.isinf(df_final[numeric_cols]).sum().sum()
if inf_check == 0:
    print("✅ There are no infinity values ​​in the data.")
else:
    print("⚠️ Warning: There are infinite values ​​in the data!")

#3. Quick statistics on important columns to make sure they make sense
print("\n---Summary Statistics)---")
cols_to_describe = ['pickup_count', 'lag_1_6h', 'rolling_mean_24h', 'target_next_6h']
print(df_final[cols_to_describe].describe().round(2))

print("\n==================================================")
print("The data is completely ready for training! 🚀")
print("==================================================")

1. اختبار دقة عمود الهدف (Target Validation Check)
✅ ممتاز! تم فحص جميع الصفوف. عمود الهدف (target_next_6h) مطابق تماماً للمستقبل بنسبة 100%.

2. الفحص العام لصحة البيانات (General Data Health)
✅ لا توجد أي قيم مفقودة (NaNs) في البيانات.
✅ لا توجد أي قيم لا نهائية (Infinity) في البيانات.

--- إحصائيات سريعة (Summary Statistics) ---
       pickup_count   lag_1_6h  rolling_mean_24h  target_next_6h
count     735085.00  735085.00         735085.00        735085.0
mean         107.96     107.93            107.94           108.0
std          306.45     306.39            258.81           306.5
min            0.00       0.00              0.00             0.0
25%            1.00       1.00              1.25             1.0
50%            4.00       4.00              5.00             4.0
75%           27.00      27.00             32.25            27.0
max         3742.00    3742.00           2277.75          3742.0

البيانات جاهزة تماماً للتدريب! 🚀


In [19]:
df_final.head(5)

,PULocationID,time_bucket_6H,pickup_count,temp_c,rain_mm,is_rain,weather_code,is_holiday,pickup_hour,day_of_week,is_weekend,lag_1_6h,lag_2_6h,lag_4_6h,rolling_mean_24h,target_next_6h
0,1,2024-01-02 00:00:00,0.0,6.3,0.0,0.0,3.0,1.0,0,1,0,0.0,2.0,0.0,0.5,0.0
1,1,2024-01-02 06:00:00,0.0,6.3,0.0,0.0,3.0,1.0,6,1,0,0.0,0.0,0.0,0.5,0.0
2,1,2024-01-02 12:00:00,0.0,6.3,0.0,0.0,3.0,1.0,12,1,0,0.0,0.0,2.0,0.0,2.0
3,1,2024-01-02 18:00:00,2.0,0.8,0.0,0.0,0.0,0.0,18,1,0,0.0,0.0,0.0,0.5,0.0
4,1,2024-01-03 00:00:00,0.0,0.8,0.0,0.0,0.0,0.0,0,2,0,2.0,0.0,0.0,0.5,0.0


In [20]:
import os

# Specify the save path and new file name
save_path = "/content/drive/MyDrive/Full_Taxi_data/processed_taxi_data_ML1(6h).parquet"

print("Saving the processed data to Google Drive...")

# Save the data in parquet format
df_final.to_parquet(save_path, index=False)

print(f"✅ Data was successfully saved to path:\n{save_path}")

# To verify that the file actually exists
if os.path.exists(save_path):
    file_size_mb = os.path.getsize(save_path) / (1024 * 1024)
    print(f"File size approx: {file_size_mb:.2f} MB")

جاري حفظ البيانات المعالجة إلى Google Drive...
✅ تم حفظ البيانات بنجاح في المسار:
/content/drive/MyDrive/Full_Taxi_data/processed_taxi_data_ML1(6h).parquet
حجم الملف تقريباً: 9.56 MB


In [21]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("1. Preparing Features and Target...")
# Specify the columns that the model will study (X)
features = [
    'PULocationID', # The area identifier is very important for the model to know where we are
    'pickup_hour', # hour
    'day_of_week', # day of the week
    'is_weekend', # weekend
    'temp_c', # temperature
    'rain_mm', # amount of rain
    'is_rain', # Is it raining
    'weather_code', # weather code
    'is_holiday', # Is today a public holiday?
    'lag_1_6h', # Request in the previous 6 hours
    'lag_2_6h', # Order 12 hours in advance
    'lag_4_6h', # Order 24 hours in advance
    'rolling_mean_24h' # Average demand in the last 24 hours (momentum)
]

target = 'target_next_6h'

# Define categorical variables so that LightGBM can handle them intelligently
categorical_features = [
    'PULocationID', 'pickup_hour', 'day_of_week',
    'is_weekend', 'is_rain', 'weather_code', 'is_holiday'
]

# Ensure that categorical variables are converted to type 'category' (a prerequisite for LightGBM)
for col in categorical_features:
    df_final[col] = df_final[col].astype('category')

print("2. Temporal Train/Test Split)...")
# Arrange data chronologically for confirmation
df_final = df_final.sort_values('time_bucket_6H').reset_index(drop=True)

# Calculate the cut-off point (e.g. 80% training and 20% testing)
split_index = int(len(df_final) * 0.80)
split_date = df_final.loc[split_index, 'time_bucket_6H']
print(f"To be trained on data before: {split_date}")
print(f"Will test on data after: {split_date}")

# Actual partition
train_data = df_final.iloc[:split_index]
test_data = df_final.iloc[split_index:]

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print(f"Training data size: {X_train.shape}")
print(f"Test data size: {X_test.shape}")

1. تجهيز الميزات (Features) والهدف (Target)...
2. التقسيم الزمني للبيانات (Temporal Train/Test Split)...
سيتم التدريب على البيانات قبل: 2025-07-14 00:00:00
سيتم الاختبار على البيانات بعد: 2025-07-14 00:00:00
حجم بيانات التدريب: (588068, 13)
حجم بيانات الاختبار: (147017, 13)


In [25]:
!pip install optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 12.6 MB/s eta 0:00:00


In [26]:
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

print("1. Start searching for the best settings with Optuna...")

# Definition of the search function (Objective Function)
def objective(trial):
    # Search space (Hyperparameter Space)
    param = {
        # Experiment with different loss functions ('huber' is excellent for dealing with outliers and reducing RMSE)
        'objective': trial.suggest_categorical('objective', ['huber', 'regression', 'regression_l1']),
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 6, 15),
        'num_leaves': trial.suggest_int('num_leaves', 31, 120),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1 # To prevent too many annoying details from being printed while searching
    }

    # Building the initial model
    model = lgb.LGBMRegressor(**param)

    # Initial model training
    model.fit(
        X_train,
        y_train,
        categorical_feature=categorical_features,
        eval_set=[(X_test, y_test)],
        eval_metric='rmse',
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
    )

    # Prediction and error calculation (we want to reduce the RMSE)
    preds = model.predict(X_test)
    preds = np.maximum(0, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return rmse

# Create an Optuna (Study)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20) # He will try 20 different combinations

print("\n2. Best settings found!")
best_params = study.best_params
print(best_params)

print("\n3. Building the final model and training it using the best settings...")
# Added fixed settings for better
best_params['random_state'] = 42
best_params['n_jobs'] = -1

final_model = lgb.LGBMRegressor(**best_params)

final_model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features,
    eval_set=[(X_test, y_test)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

print("\n4. Evaluation and extraction of results (Final Model Evaluation)...")
# Predict results
y_pred = final_model.predict(X_test)

#Get rid of negative expectations
y_pred = np.maximum(0, y_pred)

# Final calculation of accuracy metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"✅ Mean Absolute Error (MAE): {mae:.2f} excursion")
print(f"✅ Root Mean Square Error (RMSE): {rmse:.2f} excursion")
print(f"✅ Interpretation ratio (R-squared): {r2:.4f}")

# A quick look at the forecast
results_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': np.round(y_pred, 0)
}).head(10)

print("\n--- Sample forecast (first 10 periods in test data)---")
print(results_df)

[I 2026-03-03 03:33:17,014] A new study created in memory with name: no-name-6d85b0df-a734-4ae4-bc84-914cca0a0130


1. بدء البحث عن أفضل الإعدادات باستخدام Optuna...


[I 2026-03-03 03:33:51,548] Trial 0 finished with value: 305.36780676614654 and parameters: {'objective': 'huber', 'n_estimators': 400, 'learning_rate': 0.015535001330260543, 'max_depth': 10, 'num_leaves': 92, 'subsample': 0.7444415274050049, 'colsample_bytree': 0.9609273998973404}. Best is trial 0 with value: 305.36780676614654.
[I 2026-03-03 03:34:49,877] Trial 1 finished with value: 52.4094279907284 and parameters: {'objective': 'regression_l1', 'n_estimators': 700, 'learning_rate': 0.029638641145791275, 'max_depth': 15, 'num_leaves': 65, 'subsample': 0.9903735650987171, 'colsample_bytree': 0.8939237571995247}. Best is trial 1 with value: 52.4094279907284.
[I 2026-03-03 03:35:26,478] Trial 2 finished with value: 54.43095128259114 and parameters: {'objective': 'regression_l1', 'n_estimators': 500, 'learning_rate': 0.12613279720014267, 'max_depth': 6, 'num_leaves': 35, 'subsample': 0.9198087646994436, 'colsample_bytree': 0.895821588573345}. Best is trial 1 with value: 52.4094279907284


2. تم إيجاد أفضل الإعدادات!
{'objective': 'regression', 'n_estimators': 800, 'learning_rate': 0.043048267367262055, 'max_depth': 13, 'num_leaves': 78, 'subsample': 0.9285827889449285, 'colsample_bytree': 0.8280318066437822}

3. بناء الموديل النهائي وتدريبه باستخدام أفضل الإعدادات...
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[800]	valid_0's rmse: 38.9566	valid_0's l2: 1517.62

4. التقييم واستخراج النتائج (Final Model Evaluation)...
✅ متوسط الخطأ المطلق (MAE): 13.29 رحلة
✅ جذر متوسط مربع الخطأ (RMSE): 38.96 رحلة
✅ نسبة التفسير (R-squared): 0.9840

--- عينة من التوقعات (أول 10 فترات في بيانات الاختبار) ---
   Actual  Predicted
0     0.0        1.0
1     0.0        1.0
2    11.0        5.0
3    72.0       68.0
4     0.0        0.0
5     1.0        1.0
6    60.0       85.0
7     0.0        2.0
8     2.0        2.0
9    20.0       17.0


In [27]:
import joblib
import os

print("Saving the AI ​​model...")

# Specify the save path (we chose a name that reflects the ML 6 model)
model_save_path = "/content/drive/MyDrive/Full_Taxi_data/lgbm_demand_model_ml1(6h).pkl"

# Save the model (make sure you use the variable that holds the final model, whether model or final_model)
# If you use Optuna's final code, the model name is final_model
joblib.dump(final_model, model_save_path)

print("\n==================================================")
print(f"✅ The model was successfully saved to the path:\n{model_save_path}")
print("==================================================")

# To check the size of the saved model
if os.path.exists(model_save_path):
    model_size_mb = os.path.getsize(model_save_path) / (1024 * 1024)
    print(f"Model file size: {model_size_mb:.2f} MB")

جاري حفظ موديل الذكاء الاصطناعي...

✅ تم حفظ الموديل بنجاح في المسار:
/content/drive/MyDrive/Full_Taxi_data/lgbm_demand_model_ml1(6h).pkl
حجم ملف الموديل: 5.23 MB
